In [224]:
# %%
import boto3
import pandas as pd
from io import StringIO


In [225]:
# %%
BUCKET_NAME = "luis-sprint4"
RAW_KEY = "dados/InternacoesHospitalares.csv"                 # Arquivo original (sujo)
CLEAN_KEY = "dados_limpos/InternacoesHospitalares_limpo.csv"  # Arquivo tratado

s3 = boto3.client("s3")


In [226]:
# %%
obj = s3.get_object(Bucket=BUCKET_NAME, Key=RAW_KEY)

# CSV UTF-8 separado por vírgula
df = pd.read_csv(obj["Body"], sep=",", encoding="utf-8")

print("Prévia do arquivo bruto:")
df.head()


Prévia do arquivo bruto:


,,,NOME DA BASE: Internações Hospitalares;;;;
DESCRIÇÃO DA BASE: Número de internações hospitalares por especialidade,município de residência,idade,sexo do paciente;;;;
PERÍODO DE REFERÊNCIA: Ano de 2023;;;;,NaN,NaN,NaN
;;;;,NaN,NaN,NaN
Data/Hora_ Internação;Especialidade;Município;Idade;Sexo,NaN,NaN,NaN
01/01/2023 01:12;MEDICINA INTENSIVA;NATAL;42;FEMININO,NaN,NaN,NaN


In [227]:
obj = s3.get_object(Bucket=BUCKET_NAME, Key=RAW_KEY)

# Pular as primeiras 4 linhas descritivas
df = pd.read_csv(obj["Body"], sep=";", encoding="utf-8", skiprows=4)

print("Colunas lidas do CSV:")
print(df.columns.tolist())
df.head()


Colunas lidas do CSV:
['Data/Hora_ Internação', 'Especialidade', 'Município', 'Idade', 'Sexo']


,Data/Hora_ Internação,Especialidade,Município,Idade,Sexo
0,01/01/2023 01:12,MEDICINA INTENSIVA,NATAL,42,FEMININO
1,01/01/2023 05:09,CARDIOLOGIA,Macaíba,45,MASCULINO
2,01/01/2023 10:26,NEUROLOGIA,JARDIM DO SERIDÓ,71,FEMININO
3,02/01/2023 07:19,ONCOLOGIA CLÍNICA,NATAL,63,MASCULINO
4,02/01/2023 07:54,CIRURGIA VASCULAR,NATAL,66,FEMININO


In [228]:
# Renomear colunas para nomes padronizados (sem acento e em minúsculo)
df = df.rename(columns={
    "Data/Hora_ Internação": "data_internacao",
    "Especialidade": "especialidade",
    "Município": "municipio",
    "Idade": "idade",
    "Sexo": "sexo"
})

# Datas → timestamp padrão
df["data_internacao"] = pd.to_datetime(
    df["data_internacao"], errors="coerce", dayfirst=True
)

# Idade → inteiro
df["idade"] = pd.to_numeric(df["idade"], errors="coerce").fillna(0).astype(int)

# Sexo → M/F
df["sexo"] = df["sexo"].str.upper().str.strip().replace({
    "MASCULINO": "M",
    "FEMININO": "F"
})

print("Prévia após limpeza:")
df.head()


Prévia após limpeza:


,data_internacao,especialidade,municipio,idade,sexo
0,2023-01-01 01:12:00,MEDICINA INTENSIVA,NATAL,42,F
1,2023-01-01 05:09:00,CARDIOLOGIA,Macaíba,45,M
2,2023-01-01 10:26:00,NEUROLOGIA,JARDIM DO SERIDÓ,71,F
3,2023-01-02 07:19:00,ONCOLOGIA CLÍNICA,NATAL,63,M
4,2023-01-02 07:54:00,CIRURGIA VASCULAR,NATAL,66,F


In [229]:
# %%
csv_buffer = StringIO()
df.to_csv(csv_buffer, index=False, encoding="utf-8")

s3.put_object(Bucket=BUCKET_NAME, Key=CLEAN_KEY, Body=csv_buffer.getvalue())

print(f"Arquivo limpo salvo em s3://{BUCKET_NAME}/{CLEAN_KEY}")


Arquivo limpo salvo em s3://luis-sprint4/dados_limpos/InternacoesHospitalares_limpo.csv
